# Week 7 – Multi-task prompt data (Category + Price)

Build prompt/completion pairs for **multi-task**: model outputs both category and price in one completion.

Format: `Category: <cat>. Price is $<num>.00`

Run from **repo root** with `week7` on path, or from `week7` so `pricer` is importable.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
week7_dir = root / "week7" if (root / "week7" / "pricer").exists() else root
sys.path.insert(0, str(week7_dir))
contrib_dir = week7_dir / "community_contributions" / "abdussamadbello"
if contrib_dir.exists():
    sys.path.insert(0, str(contrib_dir))

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.items import Item
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

from multitask_data import item_to_multitask_datapoint, build_multitask_prompt, build_multitask_completion

In [ ]:
LITE_MODE = True  # Use True for small dataset; False for full

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
CUTOFF = 110

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
# Build multi-task prompt/completion for train and val (rounded price), test (exact price for eval)
def build_splits():
    train_dp = [item_to_multitask_datapoint(i, tokenizer, CUTOFF, do_round=True) for i in tqdm(train)]
    val_dp = [item_to_multitask_datapoint(i, tokenizer, CUTOFF, do_round=True) for i in tqdm(val)]
    test_dp = [item_to_multitask_datapoint(i, tokenizer, CUTOFF, do_round=False) for i in tqdm(test)]
    return train_dp, val_dp, test_dp

train_dp, val_dp, test_dp = build_splits()

In [ ]:
print("Example prompt:", train_dp[0]["prompt"][:200], "...")
print("Example completion:", train_dp[0]["completion"])

In [ ]:
# Optional: push to Hugging Face Hub (set MULTITASK_DATASET to your repo e.g. abdussamadbello/items_prompts_multitask_lite)
MULTITASK_DATASET = None  # e.g. "abdussamadbello/items_prompts_multitask_lite"

if MULTITASK_DATASET:
    DatasetDict({
        "train": Dataset.from_list(train_dp),
        "validation": Dataset.from_list(val_dp),
        "test": Dataset.from_list(test_dp),
    }).push_to_hub(MULTITASK_DATASET)
    print(f"Pushed to {MULTITASK_DATASET}")
else:
    print("Set MULTITASK_DATASET to push to Hub. Data is in train_dp, val_dp, test_dp.")